# R2lab monolithic OAI gNB with a Benetel RU (O-RAN 7.2)

Single OAI gNB on **sopnode-f3** driving the Benetel RAN550 **benetel1** over the O-RAN 7.2 fronthaul (DPDK on SR-IOV VFs, PTP time synchronization), with `qhat01`, `qhat02`, `qhat03` as physical UEs in R2lab.

The radio and fronthaul parameters are those of the 5g_ansible Benetel setup: https://github.com/sopnode/oai5g-rru/blob/main/rru/gnb-config-benetel.yaml and `rru/benetel1.env`.

First, we install the required dependencies:

## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal servers:

In [ ]:
%pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
%pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
! slices auth login

Install IFRAN:

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../../..").resolve()
# Refresh upstream even when its package version number has not changed.
# O-RAN 7.2 node setup (K8SManagement.setup_fhi72) is on the IFRAN fhi72 branch.
%pip install --upgrade --force-reinstall --no-deps "ifran @ git+https://github.com/ifran-project/ifran.git@fhi72"
# CSI support is in this repository (branch csi): install it in editable mode.
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


Install the Ansible dependencies:

In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r ../../../../ansible/requirements.yaml

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

IFRAN keeps the logical network description separate from the deployment-specific choices and evolving deployment state, so the same experiment can be adapted to different implementations or testbeds.

To achieve that, it uses two separate models:
- The Post-5G metamodel describing the entire network and its relationships using in pure 3GPP-standard way
- The deployment model describing implementation choices, target nodes, provider settings, and operational state

To deploy, you invoke lifecycle managers for the work you need. For example, the `ProvisioningManagement` class reserves or creates resources, `KubernetesManagement` prepares nodes and forms a cluster, `CoreManagement`, `RANManagement`, `RICManagement`, and `UEManagement` prepare and install workloads on their respective 5G components.

These stages are composable management operations, not necessarily one automatic "deploy everything" command.

Each stage exposes its particular set of actions that run the corresponding Ansible roles and Helm actions, often concurrently where possible.

In [ ]:
from metamodel_3gpp.models.metamodel import MetaModel

# Radio parameters of the 5g_ansible / oai5g-rru Benetel gNB (rru/gnb-config-benetel.yaml):
# n78, 100 MHz (273 PRB), SSB 630048, pointA 626772, TDD 2.5 ms (3 DL slots + 6 DL symbols, 1 UL slot + 4 UL symbols).
metamodel = MetaModel.from_yaml(
"""
rus:
  benetel1:
    type: real
    timeSource: external
    clockSource: external
    numberOfTxPorts: 4
    numberOfRxPorts: 4
    txAttenuationDB: 0
    rxAttenuationDB: 0
    rxGainDB: 75
ran:
  components:
    gnb:
      type: gnb
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 100
          pointAFrequencyArfcn: 626772
          ssbArfcn: 630048
          ssbPbchBlockPowerDbm: -15
          prachConfigurationIndex: 152
          msg1FrequencyStartPrb: 0
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 3
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 1
            nrofUplinkSymbols: 4
        nrCellId: 1
        physicalCellId: 0
        ru: benetel1
        gnbId: 1
ric:
  components: {}
ues:
  qhat01:
    type: cots
    config:
      supi: '001010000000006'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - gnb
  qhat02:
    type: cots
    config:
      supi: '001010000000007'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - gnb
  qhat03:
    type: cots
    config:
      supi: '001010000000008'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - gnb
"""
)


To see the full content of this Meta Model:

In [ ]:
print(metamodel.to_yaml(exclude_defaults=False))

Next, to get the deployment model, we need initiate a `DeploymentModel` object.

The easiest way to do so is by providing a YAML input that describes our deployment.

`sopnode-f1` is the Kubernetes master and runs the core, `sopnode-f3` runs the gNB, both provisioned by POS (the `legacy_pos` provisioner remains declared but no node uses it). The `fhi72` block of `sopnode-f3` describes its O-RAN 7.2 setup: fronthaul PF and its VFs (one per C/U plane, with the DU MAC and the RU VLAN), housekeeping CPUs, NIC IRQ pinning and PTP.

For that, it is necessary that you change the values in between `<...>` to contain your actual [R2Lab](https://r2lab.inria.fr/tuto-010-registration.md) credentials (your R2Lab slice name, email, and password).

In [ ]:
from ifran.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    extra_path: ''
  k8s:
    master_node: sopnode-f1
provisioners:
  pos:
    type: pos
    params:
      attempt_reservation: false
  legacy_pos:
    type: legacy_pos
    params:
      attempt_reservation: false
      central_pos_ssh_info:
        hostname: duckburg.net.cit.tum.de
        port: 10022
        user: <YOUR_SLICES_USERNAME>
  r2lab:
    type: r2lab
    params:
      r2lab_slice: <YOUR_R2LAB_SLICE>
      reservation_params:
        email: <YOUR_R2LAB_EMAIL>
        password: <YOUR_R2LAB_PASSWORD>
nodes:
  sopnode-f1:
    provisioner: pos
    roles:
    - core
  sopnode-f3:
    provisioner: pos
    roles:
    - ran
    provisioning_params:
      # 5g_ansible boot_parameters_rt_f3: housekeeping CPUs 0-5 (= fhi72 reserved_cpus), 40 x 1G hugepages
      boot_params: net.ifnames=1 biosdevname=0 isolcpus=managed_irq,6-63 nohz_full=6-63
        nohz=on rcu_nocbs=6-63 kthread_cpus=0-5 irqaffinity=0-5 rcu_nocb_poll
        selinux=0 enforcing=0 intel_pstate=disable nosoftlockup
        hugepagesz=1G hugepages=40 hugepagesz=2M hugepages=0 default_hugepagesz=1G
        mitigations=off intel_iommu=on processor.max_cstate=1 idle=poll
        intel_idle.max_cstate=0 iommu=pt skew_tick=1 tsc=nowatchdog
        nmi_watchdog=0 softlockup_panic=0 audit=0 mce=off nosmt
    fhi72:
      fronthaul_interface: ens15f1np1
      # VF order = fhi_72 du_addr order (C-plane, then U-plane)
      vfs:
      - resource_name: ruvfioc
        mac: 00:11:22:33:44:67
        vlan: 801
      - resource_name: ruvfiou
        mac: 00:11:22:33:44:66
        vlan: 801
      reserved_cpus: 0-5
      irq_pattern: ice-|ens15f[0-3]np[0-3]
      ptp:
        interface: ens15f2np2
ran:
  implementation: oai
  components:
    gnb:
      type: gnb
      advanced_config:
        local_rf: 'no'
        tr_preference: raw_if4p5
        sl_ahead: 5
        max_pdschReferenceSignalPower: -27
        ssb_PositionsInBurst_Bitmap: 1
        initialDLBWPlocationAndBandwidth: 1099
        initialDLBWPcontrolResourceSetZero: 11
        initialDLBWPsearchSpaceZero: 0
        initialULBWPlocationAndBandwidth: 1099
        prach_msg1_FDM: 0
        zeroCorrelationZoneConfig: 0
        preambleReceivedTargetPower: -100
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB_PR: 4
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 15
        p0_NominalWithGrant: -96
        p0_nominal: -96
        pusch_TargetSNRx10: 150
        pucch_TargetSNRx10: 220
        prach_dtx_threshold: 140
        pucch0_dtx_threshold: 10
        thread_pool: 20,21,22,23,24,25,26,27,28
        fhi_72:
          # dpdk_devices: set by the gNB pod from the VFs allocated by the SR-IOV device plugin
          system_core: 0
          io_core: 1
          worker_cores:
          - 2
          dpdk_mem_size: 0
          du_addr:
          - 00:11:22:33:44:67
          - 00:11:22:33:44:66
          ru_addr:
          - 70:b3:d5:e1:5b:81
          - 70:b3:d5:e1:5b:81
          mtu: 9600
          file_prefix: fhi_72
          fh_config:
          - T1a_cp_dl: [419, 470]
            T1a_cp_ul: [285, 336]
            T1a_up: [294, 345]
            Ta4: [0, 200]
            ru_config:
              iq_width: 9
              iq_width_prach: 9
      node_name: sopnode-f3
rus:
  benetel1:
    vendor: benetel
    model: ran550
    provisioner: r2lab
ues:
  qhat01:
    type: cots
    provisioner: r2lab
  qhat02:
    type: cots
    provisioner: r2lab
  qhat03:
    type: cots
    provisioner: r2lab
"""
)


> *IMPORTANT NOTE*: Make sure your R2Lab credentials are not shared publicly!

By default, the metamodel and deployment model will use Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
and one 5G slice (`internet`) with one DNN (`internet`).

## **1.**   Provisioning nodes:

In [ ]:
from ifran.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await prov.provision_nodes()

## **2.**   Kubernetes deployment:

In [ ]:
from ifran.deployment import K8SManagement

k8s = K8SManagement(prov)

In [ ]:
await k8s.deploy()

### O-RAN 7.2 fronthaul setup of sopnode-f3

Once `sopnode-f3` is in the cluster: VFs created on `ens15f1np1` and bound to `vfio-pci`, NIC IRQs pinned on the housekeeping CPUs, kubelet static CPU manager (`full-pcpus-only`), SR-IOV device plugin (resources `openshift.io/ruvfioc` and `openshift.io/ruvfiou`), and PTP (`ptp4l` slave on `ens15f2np2`, domain 24, and `phc2sys`).

In [ ]:
await k8s.setup_fhi72()


Check the node: 40 free 1G hugepages, both VFs with their MAC and VLAN 801 bound to `vfio-pci`, `ptp4l` and `phc2sys` active, and a PTP grandmaster present with a small `master_offset` (ns).

In [ ]:
# Fronthaul node state: 1G hugepages, VFs (MAC/VLAN) bound to vfio-pci, PTP (ptp4l slave, phc2sys)
check = await prov.run_node_command(
    command=["bash", "-c", " ; ".join([
        "grep -E 'HugePages_(Total|Free)|Hugepagesize' /proc/meminfo",
        "ip -d link show ens15f1np1 | grep -E '^ +vf '",
        "for vf in /sys/class/net/ens15f1np1/device/virtfn*; do "
        "echo $(basename $(readlink $vf)) $(basename $(readlink $vf/driver)); done",
        "systemctl is-active ptp4l phc2sys",
        "/usr/local/sbin/pmc -u -b 0 -d 24 'GET TIME_STATUS_NP' | grep -E 'master_offset|gmPresent|gmIdentity'",
        "true",  # print every check, even when one fails
    ])],
    nodes=["sopnode-f3"],
    become=True,
    quiet=True,
)
print(check)


In [ ]:
# The SR-IOV device plugin must advertise one VF per resource, and the kubelet the 1G hugepages
allocatable = await prov.run_node_command(
    command=["kubectl", "--kubeconfig", "/etc/kubernetes/admin.conf", "get", "node", "sopnode-f3",
             "-o", "jsonpath={.status.allocatable}"],
    nodes=["sopnode-f1"],
    become=True,
    quiet=True,
)
print(allocatable)


## **3.**   5G core deployment:

In [ ]:
from ifran.deployment import CoreManagement

core = CoreManagement(k8s)

In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()


In [ ]:
# Preserve the database and subscriber steps performed by upstream core.deploy().
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## **4.**   RAN deployment:

First, we are going to provision the remaining resources (RUs and UEs):

In [ ]:
prov_radio = ProvisioningManagement(core)
await prov_radio.provision()


> *Note*: the RU is only powered on here. benetel1 must run the OAI RU profile (`ru_config.cfg` for 100 MHz 4T4R, VLAN 801, DU MACs above), as loaded by 5g_ansible before the gNB starts.

### Deploy the monolithic gNB on sopnode-f3

> *Pending in IFRAN (fhi72 branch)*: the upstream `oai-gnb` chart has no hugepages, no SR-IOV resource claims and does not set `fhi_72.dpdk_devices` from the allocated VFs, and its configuration template does not yet render the Benetel-specific L1/RU/MAC parameters (`tx_amp_backoff_dB`, `phase_compensation`, `ru_thread_core`, `L1_rx/tx_thread_core`, antenna ports, `do_CSIRS`/`do_SRS`). The cells below deploy the gNB once the `oai-gnb-fhi72` chart variant is selected by IFRAN for Benetel RUs.

In [ ]:
from ifran.deployment import RANManagement

ran = RANManagement(prov_radio)

In [ ]:
await ran.prepare()

In [ ]:
await ran.install()

Check the gNB logs:

In [ ]:
print(await ran.get_logs(quiet=True))

The gNB is deployed and ready for UE connections.

## **5.**   Connecting UEs:

In [ ]:
from ifran.deployment import UEManagement

ue = UEManagement(preceding=ran)

In [ ]:
await ue.connect()

Check the gNB logs after connecting the UEs:

In [ ]:
print(await ran.get_logs(quiet=True))

The 3 UEs are connected to the 5G network.

Let's verify their connectivity by pinging the UPF:

In [ ]:
ping = await prov_radio.run_ue_command(
    command=["ping", "-c", "6", "11.1.1.1", "-I", "wwan0"], quiet=True, flatten=False
)


In [ ]:
print(ping["qhat01"])

In [ ]:
print(ping["qhat02"])

In [ ]:
print(ping["qhat03"])

Let's disconnect the UEs for now:

In [ ]:
await ue.disconnect(force=True)

And finally save this experiment's meta model and deployment model:

In [ ]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

> *Note*: All your Ansible run logs are stored by action, component and execution timestamp in `~/ifran/ansible-runner` by default, which you can of course change using the `runner_path` parameter in the `params.ansible` section of the deployment model.